# Challenge 1: Select traces

**Eval-driven development (EDD): Define.** This track teaches EDD: define what good looks like, measure, change, compare, ship, and feed production failures back in.
You apply it to one concrete job, a model upgrade, with a repeatable procedure of seven steps (listed in challenge 8). This is **step 2: select representative traces.** It starts from one of two situations:

- **Proactive:** an agent's quality is degrading, or a new business requirement arrives. Question: would a different model fix it, and at what cost?
- **Reactive:** a mandated upgrade, such as moving from one provider's model to another's. Question: is the mandated model safe to ship?

Both begin the same way: pull real production traces. You'll work with one order-support agent. Here you connect LangSmith, seed production-style traces,
learn the `client.list_runs()` filters, and select exactly the traces you want. Challenge 2 then fetches whole run trees and redacts customer data before it leaves LangSmith.

Run each cell in order: click it, then press `Shift+Enter`.

## 1. Enter your LangSmith key

The key is written to a local `.env` file (git-ignored) and never printed. Everything in this track lands in your own LangSmith workspace.

In [ ]:
from lab_credentials import collect
collect()

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

## 2. Seed production traces

The cell below runs the agent about 26 times and traces every run into `<your project>-prod`.
The traces carry what real traffic carries: two releases (`v1.0`, `v1.1`), an `env` and a `segment`
tag, three multi-turn threads, a few errors, and customer emails and phone numbers in the inputs.

It takes about a minute.

In [ ]:
lab_sdk.seed(client)

## 3. The filters

`lab_sdk.roots()` is a thin wrapper over `client.list_runs(project_name=..., is_root=True)`. A **root run**
is one whole agent invocation; its children are the LLM calls and tool calls. `list_runs` takes plain arguments
for the common cases and a `filter` string for the rest:

| You want | Use |
|---|---|
| Only whole invocations | `is_root=True` |
| Failures | `error=True` |
| A time window | `start_time=<datetime>` |
| A tag | `filter='has(tags, "faq")'` |
| A metadata value (release, env, thread) | `filter='and(eq(metadata_key, "release"), eq(metadata_value, "v1.0"))'` |
| A span type | `run_type="tool"` |

Note that `error=True` without `is_root` finds failing **spans**, not just failing invocations: a tool can fail while the agent still answers.

In [ ]:
from collections import Counter

all_roots = lab_sdk.roots(client)
print(len(all_roots), "root runs;", "by release:", dict(Counter(lab_sdk.run_meta(r).get("release") for r in all_roots)))

since = lab_sdk.start_of_today()
errors = list(client.list_runs(project_name=PROD, error=True, start_time=since))
faq = list(client.list_runs(project_name=PROD, is_root=True, filter='has(tags, "faq")', start_time=since))
v10 = list(client.list_runs(
    project_name=PROD, is_root=True, start_time=since,
    filter='and(eq(metadata_key, "release"), eq(metadata_value, "v1.0"))'))
thread_id = f"thread-{lab_sdk.sfx()}-1"
thread = list(client.list_runs(
    project_name=PROD, is_root=True, start_time=since,
    filter=f'and(eq(metadata_key, "thread_id"), eq(metadata_value, "{thread_id}"))'))

print("failing spans :", Counter((r.run_type, r.name) for r in errors))
print("tag faq       :", len(faq))
print("release v1.0  :", len(v10))
print("thread 1      :", len(thread), "turns")

## 4. Your turn: combine filters

Select **root runs** that are tagged `refund`, shipped in release `v1.1`, did **not** error, and started today.
A reference solution is filled in below. Try writing it yourself first, then compare.

In [ ]:
# Reference solution: try writing this yourself first.
SINCE = lab_sdk.start_of_today()
runs = list(client.list_runs(
    project_name=PROD,
    is_root=True,
    error=False,
    start_time=SINCE,
    filter='and(has(tags, "refund"), eq(metadata_key, "release"), eq(metadata_value, "v1.1"))',
))

In [ ]:
assert runs, "No runs matched. Check the filter string and that the cell above ran without errors."
assert all("refund" in (r.tags or []) and lab_sdk.run_meta(r).get("release") == "v1.1" and not r.error for r in runs), \
    "A matched run is not a refund run from release v1.1. Check your filter."
from lab_state import mark_done
mark_done("/tmp/.sdk-ch1-select-ok")
print(f"{len(runs)} runs matched: refund, release v1.1, no error")

## 5. See it in LangSmith

In [ ]:
lab_links.project_link(PROD, hint="filter by tag refund, or by metadata release; the same filters you just wrote as strings")